> **Note — notebook surface is moving.** Starting with `v0.4.0`, all notebooks
> in this repository will move to the dedicated
> [`forecastability-examples`](https://github.com/example/forecastability-examples)
> sibling repository. The library itself will keep only deterministic Python
> APIs, scripts under `scripts/`, and recipe pages under `docs/recipes/`.
> See [docs/plan/v0_4_0_examples_repo_split_ultimate_plan.md](../../docs/plan/v0_4_0_examples_repo_split_ultimate_plan.md).

<!-- type: tutorial -->
# Routing Validation Showcase (v0.3.3)

This walkthrough keeps routing validation deterministic-first. The live public entry point is `run_routing_validation()` from the stable `forecastability` facade. The frozen release fixtures under `docs/fixtures/routing_validation_regression/expected/` remain the regression-stable authority for the v0.3.3 release review.

Sections:
1. The four outcomes — `pass`, `fail`, `abstain`, `downgrade`
2. Threshold margin and rule-stability — what they measure and what they do not
3. Synthetic panel walkthrough (one case per outcome)
4. Real-series panel walkthrough (one case)
5. Reading the report markdown and the bundle JSON
6. What the new `confidence_label` means for the v0.3.4 Forecast Prep Contract

Cross-links:
- Report generator: [scripts/run_routing_validation_report.py](../../scripts/run_routing_validation_report.py)
- Optional deterministic-first narration example: [examples/univariate/agents/routing_validation_agent_review.py](../../examples/univariate/agents/routing_validation_agent_review.py)
- Agent-layer contract: [docs/reference/agent_layer.md](../../docs/reference/agent_layer.md)

## Setup

Two evidence layers appear in the notebook:
- a runnable smoke-size synthetic validation bundle from `run_routing_validation(...)`
- the frozen v0.3.3 regression fixtures and optional generated report artifacts

The notebook does not reimplement routing audit logic, confidence calibration, or report rendering in local helper code.

In [32]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from forecastability import (
    RoutingPolicyAuditConfig,
    RoutingValidationCase,
    run_forecastability_fingerprint,
    run_routing_validation,
)

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 4)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "docs").exists():
    REPO_ROOT = REPO_ROOT.parent.parent

FIXTURE_DIR = REPO_ROOT / "docs" / "fixtures" / "routing_validation_regression" / "expected"
REPORT_JSON_DIR = REPO_ROOT / "outputs" / "json" / "routing_validation"
REPORT_MARKDOWN_PATH = REPO_ROOT / "outputs" / "reports" / "routing_validation" / "report.md"
REPORT_BUNDLE_PATH = REPORT_JSON_DIR / "routing_validation_bundle.json"
REPORT_MANIFEST_PATH = REPORT_JSON_DIR / "routing_validation_report_manifest.json"

AIR_PASSENGERS_PATH = REPO_ROOT / "data" / "raw" / "air_passengers.csv"
SUNSPOTS_PATH = REPO_ROOT / "data" / "processed" / "sunspots_monthly.csv"
ETTH1_PATH = REPO_ROOT / "data" / "raw" / "etth1_oht_subset.csv"

SEED = 42
SMOKE_N = 200
N_SURROGATES = 99
AIR_PASSENGERS_MAX_LAG = 12

In [33]:
config = RoutingPolicyAuditConfig()

live_bundle = run_routing_validation(
    n_per_archetype=SMOKE_N,
    random_state=SEED,
    config=config,
)

live_frame = pd.DataFrame(
    [
        {
            "case_name": case.case_name,
            "source_kind": case.source_kind,
            "outcome": case.outcome,
            "confidence_label": case.confidence_label,
            "threshold_margin": case.threshold_margin,
            "rule_stability": case.rule_stability,
            "fingerprint_penalty_count": case.fingerprint_penalty_count,
            "expected_primary_families": ", ".join(case.expected_primary_families),
            "observed_primary_families": ", ".join(case.observed_primary_families),
        }
        for case in live_bundle.cases
    ]
)

fixture_frame = pd.DataFrame(
    json.loads((FIXTURE_DIR / "synthetic_panel.json").read_text(encoding="utf-8"))
)
audit_fixture = json.loads((FIXTURE_DIR / "audit_summary.json").read_text(encoding="utf-8"))
confidence_fixture = json.loads((FIXTURE_DIR / "confidence_labels.json").read_text(encoding="utf-8"))
calibration_fixture = json.loads((FIXTURE_DIR / "calibration.json").read_text(encoding="utf-8"))

print(f"Live smoke bundle: {len(live_frame)} synthetic cases")
print(f"Frozen regression bundle: {audit_fixture['total_cases']} synthetic cases")

Live smoke bundle: 10 synthetic cases
Frozen regression bundle: 10 synthetic cases


## 1. The four outcomes — `pass`, `fail`, `abstain`, `downgrade`

The audit outcome is about agreement between expected families and observed routing families, then about policy fragility near boundaries. It is not a forecast-accuracy score.

The current frozen synthetic panel contains all four outcomes. In the pinned v0.3.3 fixtures, `white_noise` and `long_memory` emit `abstain`, so this notebook can show the full outcome surface directly from the frozen panel while keeping the live public bundle and generated report artifacts separate.

In [25]:
outcome_reference = pd.DataFrame(
    [
        {
            "outcome": "pass",
            "when_it_happens": "Expected and observed families overlap, and both margin and stability clear the audit thresholds.",
            "what_it_does_not_mean": "Not a claim that a specific downstream model is optimal.",
        },
        {
            "outcome": "fail",
            "when_it_happens": "Expected and observed families are disjoint.",
            "what_it_does_not_mean": "Not a measure of forecast error by itself.",
        },
        {
            "outcome": "abstain",
            "when_it_happens": "The routing recommendation carries no primary families.",
            "what_it_does_not_mean": "Not a failure; it means the policy has no family-level opinion.",
        },
        {
            "outcome": "downgrade",
            "when_it_happens": "Expected and observed families overlap, but the case sits too close to a threshold or is unstable under nearby perturbations.",
            "what_it_does_not_mean": "Not wrong by itself; it is a caution signal.",
        },
    ]
)

display(outcome_reference)

live_audit = pd.DataFrame(
    [
        {
            "total_cases": live_bundle.audit.total_cases,
            "passed_cases": live_bundle.audit.passed_cases,
            "failed_cases": live_bundle.audit.failed_cases,
            "downgraded_cases": live_bundle.audit.downgraded_cases,
            "abstained_cases": live_bundle.audit.abstained_cases,
        }
    ]
)

display(Markdown("**Live smoke audit counts**"))
display(live_audit)
display(Markdown("**Frozen v0.3.3 regression counts**"))
display(pd.DataFrame([audit_fixture]))

,outcome,when_it_happens,what_it_does_not_mean
0,pass,"Expected and observed families overlap, and both margin and stability clear the audit thresholds.",Not a claim that a specific downstream model is optimal.
1,fail,Expected and observed families are disjoint.,Not a measure of forecast error by itself.
2,abstain,The routing recommendation carries no primary families.,Not a failure; it means the policy has no family-level opinion.
3,downgrade,"Expected and observed families overlap, but the case sits too close to a threshold or is unstable under nearby pertu...",Not wrong by itself; it is a caution signal.


**Live smoke audit counts**

,total_cases,passed_cases,failed_cases,downgraded_cases,abstained_cases
0,10,2,1,7,0


**Frozen v0.3.3 regression counts**

,abstained_cases,downgraded_cases,failed_cases,passed_cases,total_cases
0,2,6,1,1,10


## 2. Threshold margin and rule-stability — what they measure and what they do not

`threshold_margin` is the smallest normalized distance from the fingerprint to any active routing threshold. Small values mean the case sits near a routing boundary.

`rule_stability` is the fraction of nearby corner-plus-center perturbations that keep the same primary-family decision. Small values mean a small movement in the fingerprint can flip the route.

Neither metric measures forecasting quality. Both metrics measure policy fragility.

In [26]:
threshold_table = pd.DataFrame(
    [
        {"signal": "tau_margin", "value": config.tau_margin, "role": "Minimum threshold margin for a confident pass."},
        {"signal": "tau_margin_medium", "value": config.tau_margin_medium, "role": "Minimum threshold margin for medium confidence."},
        {"signal": "tau_stable", "value": config.tau_stable, "role": "Minimum rule-stability score for pass."},
        {"signal": "tau_stable_high", "value": config.tau_stable_high, "role": "High-confidence stability floor."},
        {"signal": "tau_stable_medium", "value": config.tau_stable_medium, "role": "Medium-confidence stability floor."},
        {"signal": "perturbation_radius", "value": config.perturbation_radius, "role": "Radius used for the deterministic rule-stability grid."},
    ]
)

display(threshold_table)

metric_focus = fixture_frame[
    [
        "case_name",
        "outcome",
        "confidence_label",
        "threshold_margin",
        "rule_stability",
        "fingerprint_penalty_count",
    ]
].sort_values(["threshold_margin", "rule_stability"], ascending=[True, False])

display(metric_focus)

print(
    "Pinned weak_seasonal_near_threshold calibration:",
    f"amplitude={calibration_fixture['calibrated_amplitude']},",
    f"threshold_margin={calibration_fixture['threshold_margin_at_calibration']:.6f},",
    f"target_band=[{calibration_fixture['target_range_low']:.3f}, {calibration_fixture['target_range_high']:.3f}]",
)

,signal,value,role
0,tau_margin,0.05,Minimum threshold margin for a confident pass.
1,tau_margin_medium,0.02,Minimum threshold margin for medium confidence.
2,tau_stable,0.80,Minimum rule-stability score for pass.
3,tau_stable_high,0.95,High-confidence stability floor.
4,tau_stable_medium,0.75,Medium-confidence stability floor.
5,perturbation_radius,0.05,Radius used for the deterministic rule-stability grid.


,case_name,outcome,confidence_label,threshold_margin,rule_stability,fingerprint_penalty_count
8,exogenous_driven,downgrade,low,0.0004,0.6,2
4,nonlinear_mixed,fail,low,0.0049,0.6,1
7,mediated_low_directness,downgrade,low,0.0071,0.6,2
9,low_directness_high_penalty,downgrade,low,0.0139,0.6,1
1,ar1,downgrade,low,0.0154,0.6,1
3,weak_seasonal_near_threshold,downgrade,low,0.0174,1.0,0
0,white_noise,abstain,abstain,0.0300,1.0,0
6,long_memory,abstain,abstain,0.0300,1.0,0
5,structural_break,downgrade,medium,0.0477,1.0,0
2,seasonal,pass,high,0.1526,1.0,0


Pinned weak_seasonal_near_threshold calibration: amplitude=1.98, threshold_margin=0.017444, target_band=[0.010, 0.025]


## 3. Synthetic panel walkthrough (one case per outcome)

The release fixtures are the cleanest way to inspect the synthetic panel because they are version-pinned and drift-protected. The table below uses one frozen case for each outcome: `seasonal` for `pass`, `nonlinear_mixed` for `fail`, `weak_seasonal_near_threshold` for `downgrade`, and `white_noise` for `abstain`.

The live smoke bundle remains useful for interactive execution, but the frozen fixture stays authoritative for the release review.

In [34]:
synthetic_examples = []
for case_name in [
    "seasonal",
    "nonlinear_mixed",
    "weak_seasonal_near_threshold",
    "white_noise",
]:
    row = fixture_frame.loc[fixture_frame["case_name"] == case_name].iloc[0]
    synthetic_examples.append(
        {
            "source": "frozen synthetic fixture",
            "case_name": row["case_name"],
            "outcome": row["outcome"],
            "confidence_label": row["confidence_label"],
            "threshold_margin": row["threshold_margin"],
            "rule_stability": row["rule_stability"],
            "expected_primary_families": ", ".join(row["expected_primary_families"]),
            "observed_primary_families": ", ".join(row["observed_primary_families"]) or "(none)",
        }
    )

display(pd.DataFrame(synthetic_examples))

display(Markdown("**Runnable smoke-size public bundle**"))
display(
    live_frame[
        [
            "case_name",
            "outcome",
            "confidence_label",
            "threshold_margin",
            "rule_stability",
        ]
    ].sort_values(["outcome", "case_name"])
 )

,source,case_name,outcome,confidence_label,threshold_margin,rule_stability,expected_primary_families,observed_primary_families
0,frozen synthetic fixture,seasonal,pass,high,0.1526,1.0,"harmonic_regression, seasonal_naive, tbats","harmonic_regression, seasonal_naive, tbats"
1,frozen synthetic fixture,nonlinear_mixed,fail,low,0.0049,0.6,"neural, nonlinear_regression","arima, ets, linear_state_space"
2,frozen synthetic fixture,weak_seasonal_near_threshold,downgrade,low,0.0174,1.0,"harmonic_regression, seasonal_naive, tbats","harmonic_regression, seasonal_naive, tbats"
3,frozen synthetic fixture,white_noise,abstain,abstain,0.0300,1.0,"naive, seasonal_naive",(none)


**Runnable smoke-size public bundle**

,case_name,outcome,confidence_label,threshold_margin,rule_stability
1,ar1,downgrade,low,0.0112,0.6
6,long_memory,downgrade,medium,0.0300,1.0
9,low_directness_high_penalty,downgrade,low,0.0012,1.0
7,mediated_low_directness,downgrade,low,0.0138,1.0
5,structural_break,downgrade,low,0.0018,0.6
3,weak_seasonal_near_threshold,downgrade,medium,0.0300,1.0
0,white_noise,downgrade,medium,0.0300,1.0
4,nonlinear_mixed,fail,high,0.1583,1.0
8,exogenous_driven,pass,medium,0.0754,1.0
2,seasonal,pass,high,0.0575,1.0


## 4. Real-series panel walkthrough (one case)

The real-panel manifest currently names three cases: `air_passengers`, `sunspots_monthly`, and `etth1_oht_subset`. For a clean checkout, the bundled `air_passengers` series is the safest runnable example here because `sunspots_monthly` is download-on-demand and `etth1_oht_subset` is shorter than the current fingerprint minimum length.

This section fingerprints the bundled `air_passengers` member through the public facade, then conditionally reads the audited real-panel output from the generated report bundle when that artifact is present.

In [28]:
real_panel_overview = pd.DataFrame(
    [
        {
            "case_name": "air_passengers",
            "source": "bundled",
            "path": str(AIR_PASSENGERS_PATH.relative_to(REPO_ROOT)),
            "available_now": AIR_PASSENGERS_PATH.exists(),
            "row_count": len(pd.read_csv(AIR_PASSENGERS_PATH)),
            "expected_primary_families": "ets, arima",
        },
        {
            "case_name": "sunspots_monthly",
            "source": "download-on-demand",
            "path": str(SUNSPOTS_PATH.relative_to(REPO_ROOT)),
            "available_now": SUNSPOTS_PATH.exists(),
            "row_count": len(pd.read_csv(SUNSPOTS_PATH)) if SUNSPOTS_PATH.exists() else None,
            "expected_primary_families": "arima, regression, nonlinear_regression",
        },
        {
            "case_name": "etth1_oht_subset",
            "source": "bundled",
            "path": str(ETTH1_PATH.relative_to(REPO_ROOT)),
            "available_now": ETTH1_PATH.exists(),
            "row_count": len(pd.read_csv(ETTH1_PATH)),
            "expected_primary_families": "regression, nonlinear_regression",
        },
    ]
)

display(real_panel_overview)

air_passengers = pd.read_csv(AIR_PASSENGERS_PATH)["passengers"].to_numpy()
air_bundle = run_forecastability_fingerprint(
    air_passengers,
    target_name="air_passengers",
    max_lag=AIR_PASSENGERS_MAX_LAG,
    n_surrogates=N_SURROGATES,
    random_state=SEED,
)

air_summary = pd.DataFrame(
    [
        {
            "case_name": "air_passengers",
            "series_length": air_passengers.size,
            "information_structure": air_bundle.fingerprint.information_structure,
            "information_horizon": air_bundle.fingerprint.information_horizon,
            "information_mass": air_bundle.fingerprint.information_mass,
            "confidence_label": air_bundle.recommendation.confidence_label,
            "observed_primary_families": ", ".join(air_bundle.recommendation.primary_families),
            "expected_primary_families_from_manifest": "ets, arima",
        }
    ]
)

display(Markdown("**Bundled panel member run through the public fingerprint surface**"))
display(air_summary)

,case_name,source,path,available_now,row_count,expected_primary_families
0,air_passengers,bundled,data/raw/air_passengers.csv,True,144.0,"ets, arima"
1,sunspots_monthly,download-on-demand,data/processed/sunspots_monthly.csv,False,NaN,"arima, regression, nonlinear_regression"
2,etth1_oht_subset,bundled,data/raw/etth1_oht_subset.csv,True,60.0,"regression, nonlinear_regression"


**Bundled panel member run through the public fingerprint surface**

,case_name,series_length,information_structure,information_horizon,information_mass,confidence_label,observed_primary_families,expected_primary_families_from_manifest
0,air_passengers,144,monotonic,12,0.9585,medium,"arima, ets, linear_state_space","ets, arima"


In [29]:
if REPORT_BUNDLE_PATH.exists():
    report_bundle = json.loads(REPORT_BUNDLE_PATH.read_text(encoding="utf-8"))
    real_case_frame = pd.DataFrame(
        [case for case in report_bundle["cases"] if case["source_kind"] == "real"]
    )
    if real_case_frame.empty:
        display(Markdown("Generated routing-validation bundle found, but it does not contain any audited real cases."))
    else:
        real_case_frame = real_case_frame.assign(
            expected_primary_families=real_case_frame["expected_primary_families"].apply(", ".join),
            observed_primary_families=real_case_frame["observed_primary_families"].apply(", ".join),
        )
        display(Markdown("**Audited real-panel cases from the generated bundle**"))
        display(
            real_case_frame[
                [
                    "case_name",
                    "outcome",
                    "confidence_label",
                    "threshold_margin",
                    "rule_stability",
                    "expected_primary_families",
                    "observed_primary_families",
                ]
            ]
        )
else:
    display(
        Markdown(
            "No generated routing-validation bundle was found under `outputs/json/routing_validation/`. The public report script materializes the audited real-panel cases when you run it."
        )
    )

Generated routing-validation bundle found, but it does not contain any audited real cases.

## 5. Reading the report markdown and the bundle JSON

The release report is an output surface, not a second implementation. When the report script has already been run, the markdown and JSON artifacts are the fastest way to review every case without recomputing the full bundle inside the notebook.

These reads stay conditional so the notebook still runs on a clean checkout with no pre-generated outputs.

In [30]:
if REPORT_MARKDOWN_PATH.exists():
    report_lines = REPORT_MARKDOWN_PATH.read_text(encoding="utf-8").splitlines()
    display(Markdown("**Report markdown preview**"))
    print("\n".join(report_lines[:24]))
else:
    display(
        Markdown(
            "No report markdown found at `outputs/reports/routing_validation/report.md`. Generate it with `uv run python scripts/run_routing_validation_report.py --smoke --no-real-panel` for a fast synthetic-only review, or omit those flags after preparing the real-panel inputs."
        )
    )

if REPORT_BUNDLE_PATH.exists():
    report_bundle = json.loads(REPORT_BUNDLE_PATH.read_text(encoding="utf-8"))
    bundle_case_frame = pd.DataFrame(report_bundle["cases"]).assign(
        expected_primary_families=lambda df: df["expected_primary_families"].apply(", ".join),
        observed_primary_families=lambda df: df["observed_primary_families"].apply(", ".join),
    )
    display(Markdown("**Bundle JSON preview**"))
    display(
        bundle_case_frame[
            [
                "case_name",
                "source_kind",
                "outcome",
                "confidence_label",
                "threshold_margin",
                "rule_stability",
            ]
        ].head(10)
    )
else:
    display(Markdown("No generated bundle JSON found yet."))

if REPORT_MANIFEST_PATH.exists():
    report_manifest = json.loads(REPORT_MANIFEST_PATH.read_text(encoding="utf-8"))
    display(Markdown("**Report manifest settings**"))
    display(pd.DataFrame([report_manifest["settings"]]))
    flagged_cases = pd.DataFrame(report_manifest.get("flagged_cases", []))
    if not flagged_cases.empty:
        flagged_cases = flagged_cases.assign(
            expected_primary_families=flagged_cases["expected_primary_families"].apply(", ".join),
            observed_primary_families=flagged_cases["observed_primary_families"].apply(", ".join),
        )
        display(Markdown("**Flagged cases from the manifest**"))
        display(
            flagged_cases[
                [
                    "case_name",
                    "source_kind",
                    "outcome",
                    "confidence_label",
                    "review_flag",
                ]
            ]
        )
else:
    display(Markdown("No report manifest JSON found yet."))

No report markdown found at `outputs/reports/routing_validation/report.md`. Generate it with `uv run python scripts/run_routing_validation_report.py --smoke --no-real-panel` for a fast synthetic-only review, or omit those flags after preparing the real-panel inputs.

**Bundle JSON preview**

,case_name,source_kind,outcome,confidence_label,threshold_margin,rule_stability
0,white_noise,synthetic,abstain,abstain,0.0300,1.0
1,ar1,synthetic,downgrade,low,0.0112,0.6
2,seasonal,synthetic,pass,high,0.0575,1.0
3,weak_seasonal_near_threshold,synthetic,fail,low,0.0080,1.0
4,nonlinear_mixed,synthetic,fail,high,0.1583,1.0
5,structural_break,synthetic,downgrade,low,0.0018,0.6
6,long_memory,synthetic,abstain,abstain,0.0300,1.0
7,mediated_low_directness,synthetic,downgrade,low,0.0138,1.0
8,exogenous_driven,synthetic,pass,medium,0.0754,1.0
9,low_directness_high_penalty,synthetic,downgrade,low,0.0012,1.0


**Report manifest settings**

,n_per_archetype,random_state,real_panel_enabled,real_panel_path,smoke,weak_seasonal_amplitude
0,200,42,False,synthetic-only,True,1.98


**Flagged cases from the manifest**

,case_name,source_kind,outcome,confidence_label,review_flag
0,white_noise,synthetic,abstain,abstain,"ABSTAIN, LOW-CONF"
1,ar1,synthetic,downgrade,low,"DOWNGRADE, LOW-CONF"
2,weak_seasonal_near_threshold,synthetic,fail,low,"FAIL, LOW-CONF"
3,nonlinear_mixed,synthetic,fail,high,FAIL
4,structural_break,synthetic,downgrade,low,"DOWNGRADE, LOW-CONF"
5,long_memory,synthetic,abstain,abstain,"ABSTAIN, LOW-CONF"
6,mediated_low_directness,synthetic,downgrade,low,"DOWNGRADE, LOW-CONF"
7,low_directness_high_penalty,synthetic,downgrade,low,"DOWNGRADE, LOW-CONF"


## 6. What the new `confidence_label` means for the v0.3.4 Forecast Prep Contract

The widened label set (`high`, `medium`, `low`, `abstain`) is a deterministic hand-off caution signal. It says how much routing evidence survives the policy audit, not how accurate a downstream model will be.

For v0.3.4, the intended forward use is: stronger labels permit a firmer family-level hand-off, lower labels trigger more conservative prep advice, and `abstain` means `no primary-family opinion` rather than `the series failed`.

Forward links only:
- Forecast Prep Contract plan: [docs/plan/v0_3_4_forecast_prep_contract_ultimate_plan.md](../../docs/plan/v0_3_4_forecast_prep_contract_ultimate_plan.md)
- Optional deterministic-first narration surface: [examples/univariate/agents/routing_validation_agent_review.py](../../examples/univariate/agents/routing_validation_agent_review.py)
- Agent contract: [docs/reference/agent_layer.md](../../docs/reference/agent_layer.md)

This notebook intentionally does not call any v0.3.4 API surface.

In [35]:
confidence_rows = pd.Series(confidence_fixture, name="confidence_label").rename_axis("case_name").reset_index()

display(confidence_rows.sort_values(["confidence_label", "case_name"]))
display(
    confidence_rows["confidence_label"]
    .value_counts()
    .rename_axis("confidence_label")
    .reset_index(name="case_count")
)
display(
    Markdown(
        "`confidence_label` is a routing hand-off caution signal, not a forecast-quality score. The current frozen synthetic panel includes all four labels: `high`, `medium`, `low`, and `abstain`; `abstain` marks empty-family recommendations rather than failed series."
    )
)

,case_name,confidence_label
2,long_memory,abstain
9,white_noise,abstain
6,seasonal,high
0,ar1,low
1,exogenous_driven,low
3,low_directness_high_penalty,low
4,mediated_low_directness,low
5,nonlinear_mixed,low
8,weak_seasonal_near_threshold,low
7,structural_break,medium


,confidence_label,case_count
0,low,6
1,abstain,2
2,high,1
3,medium,1


`confidence_label` is a routing hand-off caution signal, not a forecast-quality score. The current frozen synthetic panel includes all four labels: `high`, `medium`, `low`, and `abstain`; `abstain` marks empty-family recommendations rather than failed series.